# Learned routing, herding and agent overload: run everything

This notebook reproduces the studies in the repository and runs the **real-GPU experiment** on Colab.

| Part | What | Needs | Time (Colab) |
|---|---|---|---|
| 0 | Setup, tests | CPU | 5 min |
| A | Regenerate every figure and table from the committed simulation results | CPU | 3 min |
| B | Re-run the agent-overload simulations (synthetic and trace-calibrated) | CPU | 20-40 min |
| C | **Real LLM server**: agents overload a vLLM server, with and without admission control | **GPU** (T4 is enough; L4/A100 better) | 40-90 min |
| C' | The same experiment against a mock server (no GPU) | CPU | 15 min |
| D | Herding real-system smoke test (gateways, Redis, backends) | CPU | 10 min |
| E | Save everything to Drive / download | | 2 min |

Use **Runtime -> Change runtime type -> T4 GPU** for Part C. Run the cells top to bottom; every part after the setup is optional (set the flags in the next cell).
Studies: `research/README.md` (herding), `research/AGENTS_STUDY.md` (agent overload).

## 0. Setup

In [ ]:
#@title Configuration
BRANCH = "claude/ecstatic-cannon-ot4thw"
REPO = "https://github.com/harsharajkumar-273/adaptive-load-balancer-rl"
GITHUB_TOKEN = ""            # only needed if the repository is private (a personal access token with repo read access)

RUN_AGENT_SIM_RERUN = True    # Part B: re-run the agent-overload simulations (set False to only use committed results)
AGENT_GRIDS = ["agents", "agents_trace"]     # add "agents_regimes", "agents_trace_regimes", "agents_surge", "agents_retries" for everything (~+30 min)
RUN_REAL_AGENT_GPU = True     # Part C: needs a GPU runtime
RUN_REAL_AGENT_MOCK = False   # Part C': mock server, no GPU
RUN_HERDING_REALSYS_SMOKE = False   # Part D
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated; any chat/completions model that fits your GPU works
SLOTS = 8                    # concurrent sequences on the server (vLLM --max-num-seqs); this is C in the paper
GPU_LOADS = [0.6, 1.2, 2.4, 4.0]      # offered load in units of measured capacity
GPU_REPS = 2
GPU_DURATION = 240            # seconds of task arrivals per run


In [ ]:
import os, subprocess, sys, shutil
if not os.path.exists("/content/adaptive-load-balancer-rl"):
    url = REPO if not GITHUB_TOKEN else REPO.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.run(["git", "clone", "-b", BRANCH, url, "/content/adaptive-load-balancer-rl"], check=True)
os.chdir("/content/adaptive-load-balancer-rl")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip -q install -r requirements.txt -r research/requirements.txt duckdb nbformat
os.environ["PYTHONPATH"] = os.getcwd()
print("CPU cores:", os.cpu_count())
!nvidia-smi || echo "No GPU in this runtime (fine for every part except C)"

In [ ]:
# Fast tests (skips the slow real-server mock test)
!python -m pytest -q tests --deselect tests/test_agent_realsys.py::test_mock_end_to_end_collapse_and_protection

## A. Regenerate figures and tables from the committed results
The repository contains the simulation outputs (`research/results/*.csv`, 56,000 runs). This rebuilds every figure and the summary tables.

In [ ]:
!python -m research.analyze
!python -m research.analyze_agents
print(open("research/results/summary_agents.md").read()[:3000])

In [ ]:
from IPython.display import Image, display
for f in ["fig1_p99_vs_staleness", "fig15_burst_share", "fig16_trace", "figA1_overload", "figA7_trace_overload", "figA4_regimes", "figA8_trace_regimes", "figA6_fluid"]:
    p = f"research/figures/{f}.png"
    if os.path.exists(p):
        print(f); display(Image(p, width=900))

## B. Re-run the agent-overload simulations
`research/agent_sim.py`: one server, multi-step tasks, tool calls, KV eviction, timeouts, retries. `agents_trace*` resample task lengths and tool times from real coding-agent sessions (TraceLab).

In [ ]:
import time
if RUN_AGENT_SIM_RERUN:
    for g in AGENT_GRIDS:
        t = time.time()
        !python -m research.run {g}
        print(g, "done in", round(time.time() - t), "s")
    !python -m research.analyze_agents
    display(Image("research/figures/figA7_trace_overload.png", width=1000))

## C. Real LLM server (GPU)
Agents (tasks with L steps and tool calls, lengths and tool times resampled from real sessions) send growing prompts to a **vLLM** server through a gateway that applies an admission policy:

- `none`: admit everything
- `shed:16`: refuse a step when more than 16 requests wait
- `cap:48`: admit a new task only while fewer than 48 tasks are admitted
- `little`: admit while N < C (s + tau + D) / s, with s and tau measured online and D = 0.1 x timeout

The harness first measures the server's saturated throughput, then sets the client timeout to 15 step-times, runs the offered load (0.6x to 4x capacity) and records goodput, tasks lost mid-way and wasted steps. Time is measured in units of the server's step time, as in the simulator.

> **Untested on Colab.** The harness is tested against a mock server; the vLLM start-up below follows vLLM's documented CLI but I could not run it on a GPU. If vLLM does not start on a T4, try the L4/A100 runtime, `VLLM_USE_V1=0`, or SGLang (`python -m sglang.launch_server`), which exposes the same `/v1/completions` API.

In [ ]:
import subprocess, time, urllib.request, os
vllm_proc = None
if RUN_REAL_AGENT_GPU:
    !pip -q install vllm
    log = open("/content/vllm.log", "w")
    vllm_proc = subprocess.Popen(
        ["vllm", "serve", MODEL, "--port", "8000", "--max-num-seqs", str(SLOTS), "--enable-prefix-caching",
         "--max-model-len", "8192", "--gpu-memory-utilization", "0.85", "--dtype", "half"],
        stdout=log, stderr=subprocess.STDOUT)
    for i in range(120):                       # up to 10 minutes: model download + load
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2); print("vLLM is up after", i * 5, "s"); break
        except Exception:
            if vllm_proc.poll() is not None: break
            time.sleep(5)
    else:
        print("vLLM did not start in time")
    if vllm_proc.poll() is not None:
        print(open("/content/vllm.log").read()[-3000:])

In [ ]:
if RUN_REAL_AGENT_GPU:
    # 1) smoke test: ~6 minutes
    !python -m research.agent_realsys --backend vllm --url http://localhost:8000 --model {MODEL} --slots {SLOTS} --quick

In [ ]:
if RUN_REAL_AGENT_GPU:
    # 2) the real experiment; results append to research/results/agent_realsys.csv (resumable: rerun to add more)
    loads = " ".join(str(x) for x in GPU_LOADS)
    !python -m research.agent_realsys --backend vllm --url http://localhost:8000 --model {MODEL} --slots {SLOTS} --loads {loads} --reps {GPU_REPS} --duration {GPU_DURATION} --warmup 40 --policies none shed:16 cap:48 little

In [ ]:
# stop the server to free the GPU
if vllm_proc is not None:
    vllm_proc.terminate()

### C'. Same experiment against a mock server (no GPU)
The mock has `SLOTS` concurrent steps with lognormal service times. It exercises the full pipeline (gateway, policies, agents, metrics) but is **not** evidence about real LLM serving.

In [ ]:
if RUN_REAL_AGENT_MOCK:
    !python -m research.agent_realsys --backend mock --slots {SLOTS} --mock-step-sec 0.25 --loads 0.9 2.4 4.0 --reps 2 --duration 90 --warmup 15

### Plot the real-server results (Part C or C')

In [ ]:
if RUN_REAL_AGENT_GPU or RUN_REAL_AGENT_MOCK:
    !python -m research.analyze_agent_realsys
    display(Image("research/figures/figA9_agent_realsys.png", width=1100))

## D. Herding real-system smoke test
Gateways, Redis and FIFO backends on this machine (`research/realsys.py`). The full grid (2,160 runs) takes about 40 hours and is meant for your own machine; see `research/RUNNING_REALSYS.md`. This runs only the 5-minute smoke test.

In [ ]:
if RUN_HERDING_REALSYS_SMOKE:
    !apt-get -qq install -y redis-server > /dev/null
    !redis-server --port 6379 --save '' --daemonize yes
    !python -m research.realsys --quick --lanes 1

## E. Save the results

In [ ]:
!zip -qr results_bundle.zip research/results research/figures
print(os.path.getsize("results_bundle.zip") // 1024, "KB")
try:
    from google.colab import files, drive
    files.download("results_bundle.zip")          # browser download
    # drive.mount("/content/drive"); !cp results_bundle.zip /content/drive/MyDrive/
except Exception as e:
    print("not on Colab:", e)

**Send the results back** (`research/results/agent_realsys.csv` and the figure `figA9_agent_realsys.png`) and the paper's real-system section can be written from them. To keep them in the repository: download the zip, copy the CSV into `research/results/` in your clone, `git add` and push.